In [ ]:
%load_ext autoreload
%autoreload 2

import humanoid_bench

from fast_td3.actors import ActorEGNN, Actor, ActorMPNN, ActorHEPI
from fast_td3.actors.gnn.aegnn import AngleEGNN, get_edges_batch
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

import torch

from fast_td3.actors.gnn.egnn import unsorted_segment_sum

In [ ]:
num_envs = 4

envs = HumanoidBenchEnv("h1-maze-v0", num_envs, device="cuda:0")

obs = envs.reset()

n_act = envs.num_actions
n_obs = envs.num_obs if type(envs.num_obs) == int else envs.num_obs[0]

In [ ]:
# Initialize EGNN
egnn = ActorEGNN(n_obs = n_obs, n_act=n_act, num_envs=num_envs, init_scale=0.1, hidden_dim=32, act_fn="relu", device="cuda:0", n_layers=4, robot="h1", n_edge_feat=0, coords_agg="sum", batch_size=8192)
egnn.forward(obs)


In [ ]:
from fast_td3.actors.gnn.egnn_jax import EGNN
import jax.random as random
import jax.numpy as jnp

egnn_jax = EGNN(in_node_nf= 19, out_node_nf=1, hidden_nf=32, in_edge_nf=0, init_scale=0.1, n_layers=4, robot="h1", batch_size=8192)

key = random.PRNGKey(42)

obs_jndarray = jnp.array(obs.cpu().numpy())

params = egnn_jax.init(key, obs_jndarray)        
output = egnn_jax.apply(params, obs_jndarray)

In [ ]:
actor = ActorEGNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=112,
    n_layers=2,
    init_scale=0.1,
    act_fn="relu"
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)

In [ ]:
actor = ActorMPNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=96,
    latent_dim=96,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)

In [ ]:
actor = ActorHEPI(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=64,
    latent_dim=64,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)